# 1、 SummarizationMiddleware中间件
举例1：

In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {"type": "disabled"}  # 关键：在这里关闭思考模式
    }
)

In [ ]:
import os
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langchain_deepseek import ChatDeepSeek  # 需要先安装：pip install langchain-deepseek


def main():
    # 1. 初始化模型（建议从环境变量读取 API Key，避免硬编码泄露）
    model = ChatDeepSeek(
        model="deepseek-v4-flash",
        api_key=os.getenv("DEEPSEEK_API_KEY", "your-api-key-here"),
        temperature=0.7,
    )

    # 2. 构建对话消息历史（注意使用英文引号，避免语法错误）
    messages = [
        SystemMessage("你是个非常友好的AI助手"),
        HumanMessage("你好啊，我是老王，你是谁？"),
        AIMessage("你好老王，我是小王"),
        HumanMessage("好的小王，很高兴认识你"),
        AIMessage("你高兴得太早了"),
        HumanMessage("呵呵，你什么意思"),
    ]

    # 3. 创建 Agent，并挂载摘要中间件以自动压缩超长上下文
    agent = create_agent(
        model=model,
        middleware=[
            SummarizationMiddleware(
                model=model,  # 摘要生成也需要传入模型实例
                trigger=[
                    ("tokens", 100),      # 当总 token 数超过 100 时触发摘要
                    ("messages", 6),      # 当消息条数超过 6 条时触发摘要
                    ("fraction", 0.001),  # 当上下文占总窗口比例超过 0.1% 时触发摘要
                ],
                keep=("messages", 2),     # 摘要后保留最后 2 条原始消息
            )
        ],
    )

    # 4. 执行 Agent 调用
    response = agent.invoke({"messages": messages})

    # 5. 美化打印输出结果中的每一条消息
    for msg in response["messages"]:
        msg.pretty_print()


if __name__ == "__main__":
    main()

## 2、举例2测试summary_prompt参数

In [3]:
import os
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langchain_deepseek import ChatDeepSeek  # 需要先安装：pip install langchain-deepseek


def main():
    # 1. 初始化模型（建议从环境变量读取 API Key，避免硬编码泄露）
    model = ChatDeepSeek(
        model="deepseek-v4-flash",
        api_key=os.getenv("DEEPSEEK_API_KEY", "your-api-key-here"),
        temperature=0.7,
    )

    # 2. 构建对话消息历史（注意使用英文引号，避免语法错误）
    messages = [
        SystemMessage("你是个非常友好的AI助手"),
        HumanMessage("你好啊，我是老王，你是谁？"),
        AIMessage("你好老王，我是小王"),
        HumanMessage("好的小王，很高兴认识你"),
        AIMessage("你高兴得太早了"),
        HumanMessage("呵呵，你什么意思"),
    ]

    # 3. 创建 Agent，并挂载摘要中间件以自动压缩超长上下文
    agent = create_agent(
        model=model,
        middleware=[
            SummarizationMiddleware(
                model=model,  # 摘要生成也需要传入模型实例
                trigger=[
                    ("tokens", 100),      # 当总 token 数超过 100 时触发摘要
                    ("messages", 6),      # 当消息条数超过 6 条时触发摘要
                    # ("fraction", 0.001),  # 当上下文占总窗口比例超过 0.1% 时触发摘要
                ],
                keep=("messages", 2),     # 摘要后保留最后 2 条原始消息
                summary_prompt=f"对历史消息摘要，消息列表如下\n{messages}"
            )
        ],
    )

    # 4. 执行 Agent 调用
    response = agent.invoke({"messages": messages})

    # 5. 美化打印输出结果中的每一条消息
    for msg in response["messages"]:
        msg.pretty_print()


if __name__ == "__main__":
    main()

================================ Human Message =================================

Here is a summary of the conversation to date:

Error generating summary: Replacement index 0 out of range for positional args tuple
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

抱歉，刚才那句“你高兴得太早了”确实容易让人误会，我先解释一下。

你最初发来了一句话：

> Here is a summary of the conversation to date: Error generating summary: Replacement index 0 out of range for positional args tuple

这句话看起来像是在说“这里是对话摘要”，但后面跟的却是一个报错信息。我误以为你是在用这个报错来调侃“想要摘要但失败了”，所以随口接了一句“你高兴得太早了”——意思大概是“先别急着说有摘要，其实没生成出来”。

但这句话没有上下文，确实显得莫名其妙，还带点讽刺，让你感到困惑，这是我的问题，向你道歉。

如果你其实是想问这个报错是什么意思，我可以解释：

`Replacement index 0 out of range for positional args tuple` 是 Python 中格式化字符串时常见的错误。意思是你在字符串里写了类似于 `{0}` 的占位符，但后面提供的参数元组是空的，比如：

```python
"你好 {0}".format()
```

就会